# Scrape DSEBD Sector-wise Company List
This notebook provides a step-by-step guide on how to scrape the [Dhaka Stock Exchange (DSE) Sector-wise Listing](https://www.dsebd.org/by_industrylisting.php). It extracts all the sectors, navigates to each sector's sub-page to collect its symbols, and then categorizes each symbol according to its group (A, B, G, N, Z).

### Step 1: Import all required libraries
We will use `requests` for fetching the webpages, `BeautifulSoup` for parsing the HTML, `time` to add delays between requests (to be polite to the server), and `pandas` for organizing the data and saving it to a CSV.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

### Step 2: Fetch the Sectors
We use `requests` to download the main sector listing page. The sectors are found inside `<a>` tags with the class `ab1` inside `<td>` elements.

In [ ]:
base_url = 'https://www.dsebd.org/'
industry_url = base_url + 'by_industrylisting.php'
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}

print("Fetching sectors...")
response = requests.get(industry_url, headers=headers)
response.raise_for_status()

soup = BeautifulSoup(response.text, 'html.parser')

sectors = []
for a_tag in soup.find_all('a', class_='ab1'):
    sector_name = a_tag.get_text(strip=True)
    link = a_tag.get('href')
    if link and 'companylistbyindustry.php' in link:
        sectors.append({
            'name': sector_name,
            'url': base_url + link
        })

print(f"Found {len(sectors)} sectors.")
for s in sectors[:5]:
    print(f"- {s['name']}: {s['url']}")

Fetching sectors...
Found 22 sectors.
- Bank: https://www.dsebd.org/companylistbyindustry.php?industryno=11
- Cement: https://www.dsebd.org/companylistbyindustry.php?industryno=21
- Ceramics Sector: https://www.dsebd.org/companylistbyindustry.php?industryno=24
- Corporate Bond: https://www.dsebd.org/companylistbyindustry.php?industryno=26
- Debenture: https://www.dsebd.org/companylistbyindustry.php?industryno=91


### Step 3: Fetch Symbols for Each Sector
Now we loop through each sector, load its specific page, and extract the symbols listed there. The symbols typically appear in anchor `<a>` tags linking to `displayCompany.php`.

In [ ]:
parsed_data = []

print("Fetching symbols for each sector...")
for sector in sectors:
    try:
        sec_resp = requests.get(sector['url'], headers=headers)
        sec_resp.raise_for_status()
        sec_soup = BeautifulSoup(sec_resp.text, 'html.parser')
        
        for a_tag in sec_soup.find_all('a',class_='ab1', href=True):
            if 'displayCompany.php' in a_tag['href']:
                symbol = a_tag.get_text(strip=True)
                if symbol and not any(d['Symbol'] == symbol and d['Sector'] == sector['name'] for d in parsed_data):
                    parsed_data.append({'Symbol': symbol, 'Sector': sector['name']})
                    
        time.sleep(0.5)
        
    except Exception as e:
        print(f"Failed to process {sector['name']}: {e}")

df = pd.DataFrame(parsed_data)
print(f"\nExtracted {len(df)} unique symbols across {df['Sector'].nunique()} sectors.")
print(df.head(10))

Fetching symbols for each sector...

Extracted 650 unique symbols across 22 sectors.
       Symbol Sector
0      ABBANK   Bank
1   ALARABANK   Bank
2    BANKASIA   Bank
3    BRACBANK   Bank
4    CITYBANK   Bank
5   DHAKABANK   Bank
6  DUTCHBANGL   Bank
7         EBL   Bank
8    EXIMBANK   Bank
9  FIRSTSBANK   Bank


### Step 4: Fetch Categories (A, B, G, N, Z)
Next, we map each symbol to its market category. The categories can be fetched from the DSE group pages. We fetch symbols in each category using class `ab1` and update our dataframe column. Note: If a symbol is not found in any category, it will map to `None` (empty).

In [ ]:
groups = ['A', 'B', 'G', 'N', 'Z']
# Create empty column first
df['Category'] = None
category_map = {}

print("Fetching symbol categories...")
for grp in groups:
    try:
        # The category pages often end with latest_share_price_scroll_group.php?group=
        cat_url = base_url + f'latest_share_price_scroll_group.php?group={grp}'
        cat_resp = requests.get(cat_url, headers=headers)
        cat_resp.raise_for_status()
        cat_soup = BeautifulSoup(cat_resp.text, 'html.parser')
        
        # Categories typically use the same 'ab1' class for symbol links
        for a_tag in cat_soup.find_all('a', class_='ab1'):
            if 'displayCompany.php' in a_tag.get('href', ''):
                symbol = a_tag.get_text(strip=True)
                if symbol:
                    category_map[symbol] = grp
                    
        time.sleep(0.5)
    except Exception as e:
        print(f"Failed to process category {grp}: {e}")

# Map the fetched categories into the DataFrame
df['Category'] = df['Symbol'].map(category_map)

print(f"Mapped {df['Category'].notna().sum()} symbols to categories.")
print(df['Category'].value_counts(dropna=False))

Fetching symbol categories...
Mapped 397 symbols to categories.
Category
NaN    253
A      204
Z      113
B       80
Name: count, dtype: int64


### Step 5: Validate and Save to CSV
Finally, we write our completed results, which now include the categories, to the output CSV file.

In [ ]:
output_file = 'dse_symbols_by_sector.csv'
df.to_csv(output_file, index=False)
print(f"Data successfully saved to {output_file}")

# Show an overall preview
df.head()

Data successfully saved to dse_symbols_by_sector.csv


,Symbol,Sector,Category
0,ABBANK,Bank,B
1,ALARABANK,Bank,A
2,BANKASIA,Bank,A
3,BRACBANK,Bank,A
4,CITYBANK,Bank,A
